[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/manipulability/manipulability.ipynb)

# Velocity and Force Ellipsoids of an Arm

This example measures how well a robot arm can move and push at its gripper. Turning the joints at unit rates moves the gripper tip with velocities that fill an ellipsoid: long where the arm moves the tip easily, flat where it can hardly move it at all. Holding the joints with unit torques resists tip forces that fill another ellipsoid, long exactly where the first is flat. As the arm stretches out straight, the velocity ellipsoid flattens along the arm and the force ellipsoid runs off along it: a straight arm cannot move its tip outwards, but holds any force along it for free.

Each joint turns about an axis, and carried into the world by its pivot that axis is the joint's twist per unit rate, a bivector. Both ellipsoids are built from the twists alone. Motion: a twist's commutator with the tip is the tip's velocity per unit rate, a direction, and the velocity ellipsoid is the sum of those directions' dyads about the tip, a map from planes to points. Incidence: a twist joined with the tip is a plane, and a tip force within it loads the joint not at all, so the force ellipsoid is the sum of those planes' dyads, a map from points to planes. The two are each other's polar in the unit sphere about the tip: the principle of virtual work, with no Jacobian formed.

In the matrix notation of robotics the twists read as the columns of the Jacobian $J$, the velocity ellipsoid as $JJ^{\mathsf T}$ and the force ellipsoid as $(JJ^{\mathsf T})^{-1}$, the manipulability ellipsoids of Yoshikawa.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import Extensor
from examples.animation import save_figure, show_animation
from examples.mechanics.manipulability import render, scenarios
from examples.mechanics.manipulability.core import DualQuadric, Plane, Point, Quadric, Twist, arm, coordinate_planes, mv, w

np.set_printoptions(precision=4, suppress=True)

## 1. The arm and its twists

The arm is the scenegraph's: a base that yaws about z, and a shoulder, elbow and wrist that pitch about y. Each joint's generator, carried by the pivot that places the joint, is that joint's twist in the world.

In [ ]:
# Joint angles of two poses, base yaw, shoulder, elbow and wrist pitch: reaching, and nearly straight.
reaching = (0.35, 0.5, 1.2, 0.4)
nearly_straight = (0.35, 0.2, 0.15, 0.1)

In [ ]:
def arm_axes(joint_angles: tuple) -> tuple[arm.PointMap, Point, Twist]:
    """The arm's body maps, its gripper tip, and each joint's twist per unit rate in the world.

    A joint's twist is its rotation generator, carried into the world by its pivot: the base yaws
    about z, the shoulder, elbow and wrist pitch about y.
    """
    bodies, pivots = arm.robot_arm(joint_angles)
    # The top face of the gripper, on the wrist.
    tip = pivots[-1] >> mv.point([0.0, 0.0, 0.3])               # [] Point
    axes = Extensor.stack([pivots[0] >> mv.xy] + [pivot >> mv.zx for pivot in pivots[1:]])    # [joints] Twist
    return bodies, tip, axes


bodies, tip, twists = arm_axes(reaching)                                       # [bodies] Point <- Point, [] Point, [joints] Twist

In [ ]:
print("tip:", render.euclidean(tip))
for index, joint in enumerate(("base", "shoulder", "elbow", "wrist")):
    print(f"{joint} twist:\n{twists[index].coefficient_table()}")

## 2. The two ellipsoids

The tip velocity per unit rate of each joint is the twist's commutator with the tip, a direction; summing their dyads about the tip gives the velocity ellipsoid as a map from planes to points, which sends a plane to the point where it touches. Each twist joined with the tip is a plane through both, and a force at the tip within that plane runs through the joint's axis or parallel to it; across it, the joint's torque is the force read by that plane. Summing those planes' dyads gives the force ellipsoid as a map from points to planes. The unit sphere about the tip turns one into the other: the force ellipsoid is the velocity ellipsoid's polar in it.

In [ ]:
def velocity_ellipsoid(tip: Point, axes: Twist) -> DualQuadric:
    """The tip velocities reachable with unit joint rates: the velocities' dyads about the tip.

    `tip` and the joint `axes` may be in any frame, the same one for both; the ellipsoid comes out
    in that frame.
    """
    # The tip velocity per unit rate of each joint.
    velocities = axes.commutator(tip)                              # [joints] Direction
    return (velocities * (Plane & velocities)).sum() - tip * (Plane & tip)


def force_ellipsoid(tip: Point, axes: Twist) -> Quadric:
    """The tip forces resisted with unit joint torques: the dyads of the planes through each joint's
    axis and the tip, a point quadric.

    `tip` and the joint `axes` may be in any frame, the same one for both; the ellipsoid comes out
    in that frame.
    """
    # Each joint's axis joined with the tip.
    torque_free = axes & tip                                       # [joints] Plane
    return (torque_free * (torque_free & Point)).sum() - w * (w & Point)


def unit_sphere(centre: Point) -> Quadric:
    """The unit sphere about a point: the dyads of the coordinate planes through it."""
    through = coordinate_planes - w * (coordinate_planes & centre)  # [3] Plane
    return (through * (through & Point)).sum() - w * (w & Point)


velocity = velocity_ellipsoid(tip, twists)                                     # [] Point <- Plane
force = force_ellipsoid(tip, twists)                                           # [] Plane <- Point
sphere = unit_sphere(tip)                                                      # [] Plane <- Point
# The velocity ellipsoid's polar in the unit sphere about the tip.
polar = sphere(velocity(sphere(Point)))                                        # [] Plane <- Point

In [ ]:
print("force ellipsoid less the velocity ellipsoid's polar:", np.abs((force - polar).kernel).max())

## 3. Reaching, and nearly straight

Each row is one pose: the velocity ellipsoid, blue, beside the force ellipsoid, orange, both at the gripper. For display the velocity ellipsoid is grown and the force ellipsoid shrunk. Reaching, both are round enough; nearly straight, the velocity ellipsoid is a thin disk across the arm and the force ellipsoid a long cigar along it.

In [ ]:
# Display scales, the view, the pixels across each panel, and the poses drawn.
velocity_scale, force_scale = 0.25, 2.5
view, centre, extent = (18, -60), np.array([-0.9, 0.25, 1.3]), 2.2
figure_pixels = 400
poses = (reaching, nearly_straight)

In [ ]:
def ellipsoids(joint_angles: tuple):
    """The arm in one pose, its tip, and its velocity and force ellipsoids as maps from points, scaled
    for display."""
    bodies, tip, twists = arm_axes(joint_angles)
    return (bodies, tip, velocity_ellipsoid(tip, twists * velocity_scale).inverse(),
            force_ellipsoid(tip, twists * force_scale))


save_figure(render.draw_ellipsoids([ellipsoids(pose) for pose in poses], view, centre, extent, figure_pixels), "manipulability");

## 4. Around a loop through joint space

Each joint swings about a centre, the elbow never straightening fully, and the two ellipsoids turn and reshape together, each flattening where the other stretches.

In [ ]:
n_frames, frame_pixels, frame_duration = 96, 300, 60
frames = [render.frame(pose, view, centre, extent, frame_pixels) for pose in scenarios.sweep(n_frames)]
show_animation(frames, "manipulability_sweep", frame_duration)

In [ ]:
# checks
# The force ellipsoid is the velocity ellipsoid's polar in the unit sphere about the tip, in both poses.
for pose in poses:
    _, pose_tip, pose_twists = arm_axes(pose)
    pose_sphere = unit_sphere(pose_tip)
    np.testing.assert_allclose(force_ellipsoid(pose_tip, pose_twists).kernel,
                               pose_sphere(velocity_ellipsoid(pose_tip, pose_twists)(pose_sphere(Point))).kernel, rtol=1e-8, atol=1e-8)